# Lab 2: LightRAG (Graph-Enhanced Retrieval)

**학습 목표**
- 지식 그래프(Knowledge Graph) 기반 검색의 동작 원리 이해
- LLM을 이용한 엔티티/관계 추출 경험
- 벡터 검색과 그래프 탐색의 결합 효과 비교

**소요 시간**: 30-40분  
**선행 과제**: Lab 1 (Standard RAG)

In [ ]:
import sys
sys.path.insert(0, '..')

from rag_lab import EmbeddingEngine, LightRAG, load_papers
from rag_lab.evaluate import Evaluator

## 1. LightRAG 구축

Standard RAG와 달리, LightRAG는 구축 시 **LLM을 호출하여 엔티티와 관계를 추출**합니다.

→ 구축 시간이 더 오래 걸리지만, 구조화된 지식을 얻습니��.

In [ ]:
papers = load_papers()
engine = EmbeddingEngine()

# LightRAG 구축 (엔티티 추출 포함 — 1-2분 소요)
lrag = LightRAG(papers, engine)

## 2. 지식 그래프 탐색

추출된 엔티티와 관계를 살펴봅시다.

In [ ]:
# 그래프 통계
stats = lrag.get_graph_stats()
print(f"엔티티 수: {stats['total_entities']}")
print(f"관계 수:   {stats['total_relations']}")
print(f"\n엔티티 유형별 분포:")
for t, count in sorted(stats['entity_types'].items(), key=lambda x: -x[1]):
    print(f"  {t}: {count}")
print(f"\n관계 유형별 분포:")
for r, count in sorted(stats['relation_types'].items(), key=lambda x: -x[1]):
    print(f"  {r}: {count}")

In [ ]:
# 특정 엔티티의 이웃 관계
# (추출된 엔티티 이름은 그래프마다 다를 수 있음)
entities = list(lrag.entities.keys())
print("추출된 엔티티 목록:")
for e in entities[:15]:
    info = lrag.entities[e]
    print(f"  [{info['type']}] {e}: {info['description'][:60]}")

if entities:
    # 첫 번째 엔티티의 이웃 ��색
    nbr = lrag.get_entity_neighborhood(entities[0])
    print(f"\n'{entities[0]}'의 관��:")
    for rel in nbr['neighbors']['outgoing']:
        print(f"  → {rel['relation']} → {rel['target']}")
    for rel in nbr['neighbors']['incoming']:
        print(f"  ← {rel['relation']} ← {rel['source']}")

## 3. 그래프 검색 vs ��터 검색

LightRAG는 두 가지 검색을 결합합니다:
1. **벡터 검색**: Standard RAG와 동일 (유사도 기반)
2. **그래프 탐색**: 질문의 엔티티 → 1-hop 이웃 → 관련 정보

In [ ]:
# 크로스 논문 질문 — 그래프가 빛나는 영역
question = (
    "What role does employer credit checking play in labor markets, "
    "and how does it relate to information asymmetry?"
)

result = lrag.query(question)

print(f"쿼리 시간: {result.total_time:.2f}초")
print(f"그래프 컨텍스트 크기: {result.graph_context_length:,} 글자")
print(f"출처: {result.sources}")
print(f"\n답변:\n{result.answer}")

## 4. Standard RAG와 비교

같은 질문에 대해 두 방법의 답변을 비교해봅시다.

In [ ]:
from rag_lab import StandardRAG

rag = StandardRAG(papers, engine)

question = "Compare methodological approaches across human capital and auction papers"

rag_result = rag.query(question)
lrag_result = lrag.query(question)

print("=== Standard RAG ===")
print(f"시간: {rag_result.total_time:.2f}s | 출처: {rag_result.sources}")
print(rag_result.answer[:400])

print("\n=== LightRAG ===")
print(f"시간: {lrag_result.total_time:.2f}s | 출처: {lrag_result.sources}")
print(lrag_result.answer[:400])

In [ ]:
# LLM-as-Judge로 자동 평가
evaluator = Evaluator()
scores = evaluator.compare(
    question,
    {"Standard RAG": rag_result.answer, "LightRAG": lrag_result.answer},
    question_type="cross_paper_synthesis",
)
evaluator.print_comparison(scores)

## 5. 실험 과��

### 과제 A: 그래프 시각화
```python
# networkx로 지식 그래프를 시각화해보세요
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
for name, info in lrag.entities.items():
    G.add_node(name, type=info['type'])
for rel in lrag.relations:
    G.add_edge(rel['source'], rel['target'], relation=rel['relation'])

# TODO: 시각화 코드 작성
```

### 과제 B: 그래프 없이 vs 있을 때
```python
# LightRAG에서 그래프 컨텍스트를 제거하면 결과가 어떻게 변할까?
# Hint: _graph_traverse() 메서드를 빈 문자열 반환으로 오버라이드
```

### 과제 C: 엔티티 추출 프롬프트 개선
```python
# lightrag.py의 _extract_entities() 프롬프트를 수정하여
# 더 많은/적은 엔티티를 추출하게 하고 결과 비���
```